<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/simple_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [302]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader

In [303]:
!ls
file = "./100_Unique_QA_Dataset.csv"

100_Unique_QA_Dataset.csv  sample_data


In [304]:
import pandas as pd

In [305]:
df = pd.read_csv(file)
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [306]:
df.loc[0]

,0
question,What is the capital of France?
answer,Paris


In [307]:
def tokenize(text):
  text = text.lower()
  text = text.replace("?" , "")
  text = text.replace("'" , "")
  return text.split()

In [308]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [309]:
def build_vocab(row):
  token_ques = tokenize(row["question"])
  token_ans = tokenize(row['answer'])

  merge_token = token_ans+token_ques
  # print(merge_token)

  for token in merge_token:
    if token in vocab :
      pass
    else :
      vocab[token] = len(vocab)


In [310]:
vocab = {"<unk>":0}


In [311]:
df.apply(build_vocab , axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [312]:
len(vocab)

324

In [313]:
def text_to_index(text , vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else :
      indexed_text.append(vocab["<unk>"])
  return indexed_text

In [314]:
ind = text_to_index("What is the capital of France?" , vocab)
print(ind)

[2, 3, 4, 5, 6, 7]


In [315]:
class QA_dataset(Dataset):
  def __init__(self , df , vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self , index):
    ques = text_to_index(self.df.iloc[index]["question"], self.vocab)
    ans = text_to_index(self.df.iloc[index]["answer"], self.vocab)
    return torch.tensor(ques) , torch.tensor(ans)


In [316]:
dataset = QA_dataset(df , vocab)
dataset[0]

(tensor([2, 3, 4, 5, 6, 7]), tensor([1]))

In [317]:
loaded_data = DataLoader(dataset , batch_size=1 , shuffle=True)

In [318]:
class simpleRNN(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size , 50 )
    self.rnn = nn.RNN(50 , 64 , batch_first=True)
    self.fc = nn.Linear(64 , vocab_size)
  def forward(self , x):
    em = self.embedding(x)
    hidden , output = self.rnn(em)
    y_hat = self.fc(output)

    return y_hat

In [319]:
model = simpleRNN(len(vocab))

In [320]:
learning_rate = 0.001
epoch = 50

In [321]:
optimizer = optim.Adam(model.parameters() , lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

In [322]:
for epoch in range(epoch):
  total_loss = 0

  for i , (input , target) in enumerate(loaded_data):
    optimizer.zero_grad()

    output = model(input)
    # print(output.shape)

    output = output.squeeze(1)
    target = target.squeeze(1)
    loss = loss_fn(output , target)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.467857
Epoch: 2, Loss: 452.352021
Epoch: 3, Loss: 372.226459
Epoch: 4, Loss: 313.172737
Epoch: 5, Loss: 262.373788
Epoch: 6, Loss: 215.011379
Epoch: 7, Loss: 171.254105
Epoch: 8, Loss: 133.414340
Epoch: 9, Loss: 102.170329
Epoch: 10, Loss: 78.703423
Epoch: 11, Loss: 60.827651
Epoch: 12, Loss: 47.686083
Epoch: 13, Loss: 38.064529
Epoch: 14, Loss: 31.008044
Epoch: 15, Loss: 25.748152
Epoch: 16, Loss: 21.602968
Epoch: 17, Loss: 18.364926
Epoch: 18, Loss: 15.578907
Epoch: 19, Loss: 13.340878
Epoch: 20, Loss: 11.578985
Epoch: 21, Loss: 10.066893
Epoch: 22, Loss: 8.866808
Epoch: 23, Loss: 7.853300
Epoch: 24, Loss: 7.036603
Epoch: 25, Loss: 6.364991
Epoch: 26, Loss: 5.643322
Epoch: 27, Loss: 5.116991
Epoch: 28, Loss: 4.630344
Epoch: 29, Loss: 4.211272
Epoch: 30, Loss: 3.869686
Epoch: 31, Loss: 3.538014
Epoch: 32, Loss: 3.259307
Epoch: 33, Loss: 2.997390
Epoch: 34, Loss: 2.773180
Epoch: 35, Loss: 2.565395
Epoch: 36, Loss: 2.384672
Epoch: 37, Loss: 2.211889
Epoch: 38, Loss: 

In [350]:
def predict(ques , model , threshold=0.5):
  model.eval()
  number_ques = text_to_index(ques , vocab)
  tensor_ques = torch.tensor(number_ques).unsqueeze(0)

  logit = model(tensor_ques)
  pred = torch.nn.functional.softmax(logit , dim=-1)
  value , index = torch.max(pred , dim = 2)
  print(value , index)
  if value < threshold:
    print("i don't know")
  else:
     return list(vocab.keys())[index.item()]


In [353]:
predict("Who is capital of Germany" , model)

tensor([[0.9885]], grad_fn=<MaxBackward0>) tensor([[8]])


'berlin'